In [ ]:
import pandas as pd
import json
import warnings
import numpy as np
warnings.filterwarnings('ignore')

In [ ]:
# =========================================
# PROCESAMIENTO DE DATOS - STOP WEB3
# =========================================

# 1. Carga de Datos
url = "../estadistica_stop/ESTADISTICA_DELITO.csv"
df = pd.read_csv(url)

In [ ]:
# Export Santiago para pruebas
dfSantiago = df[df["codcom"] == 13101]
dfSantiago.to_csv("santiago.csv", index=False)

In [ ]:
# =========================================
# 2. CALCULAR TOTALES POR COMUNA/SEMANA
# =========================================
totales = df.groupby(['codcom', 'id_semana'], as_index=False)['frecuencia'].sum()
totales['delito'] = 'Total'
dim_tiempo = df[['id_semana', 'semana_detalle', 'fecha']].drop_duplicates()
totales = totales.merge(dim_tiempo, on='id_semana', how='left')
totales = totales[df.columns]
df = pd.concat([df, totales], ignore_index=True)

# =========================================
# 3. PREPARACIÓN TEMPORAL
# =========================================
df['fecha'] = pd.to_datetime(df['fecha'])
df['año'] = df['fecha'].dt.year
df['mes'] = df['fecha'].dt.month
df['semana_numero'] = df['semana_detalle'].apply(lambda x: int(x[7:9]))

# CRÍTICO: Ordenar antes de cualquier cálculo de ventana
df = df.sort_values(['delito', 'codcom', 'id_semana']).reset_index(drop=True)

# =========================================
# 4. VARIABLES BASE (Actual, Anterior, Delta)
# =========================================
df['casos_semana_actual'] = df['frecuencia']
df['casos_semana_anterior'] = df.groupby(['delito', 'codcom'])['frecuencia'].shift(1)
df['delta'] = df['casos_semana_actual'] - df['casos_semana_anterior']

# =========================================
# 5. ACUMULADOS
# =========================================
df['acumulado_anual'] = df.groupby(['delito', 'codcom', 'año'])['frecuencia'].cumsum()
df['acumulado_total'] = df.groupby(['delito', 'codcom'])['frecuencia'].cumsum()

# 5b. Acumulado Año Anterior (Misma Semana)
df_prev_acum = df[['delito', 'codcom', 'año', 'semana_numero', 'acumulado_anual']].copy()
df_prev_acum['año'] = df_prev_acum['año'] + 1
df_prev_acum = df_prev_acum.rename(columns={'acumulado_anual': 'acumulado_anual_anterior'})
df = df.merge(df_prev_acum, on=['delito', 'codcom', 'año', 'semana_numero'], how='left')

# =========================================
# 6. MEDIAS MÓVILES (Corregido con transform)
# =========================================
df['media_movil_4s'] = df.groupby(['delito', 'codcom'])['frecuencia'].transform(
    lambda x: x.rolling(4, min_periods=1).mean()
)
df['media_movil_8s'] = df.groupby(['delito', 'codcom'])['frecuencia'].transform(
    lambda x: x.rolling(8, min_periods=1).mean()
)

# =========================================
# 7. ESTADÍSTICAS HISTÓRICAS (Corregido con transform)
# =========================================
df['promedio_hist'] = df.groupby(['delito', 'codcom'])['frecuencia'].transform(
    lambda x: x.expanding().mean()
)
df['std_hist'] = df.groupby(['delito', 'codcom'])['frecuencia'].transform(
    lambda x: x.expanding().std()
)
df['max_hist'] = df.groupby(['delito', 'codcom'])['frecuencia'].transform(
    lambda x: x.expanding().max()
)

# =========================================
# 8. ESTADÍSTICAS AÑO ANTERIOR
# =========================================
# Calcular stats por año y luego hacer shift temporal
df['promedio_hist_anual'] = df.groupby(['delito', 'codcom', 'año'])['frecuencia'].transform(
    lambda x: x.expanding().mean()
)
df['std_hist_anual'] = df.groupby(['delito', 'codcom', 'año'])['frecuencia'].transform(
    lambda x: x.expanding().std()
)
df['max_hist_anual'] = df.groupby(['delito', 'codcom', 'año'])['frecuencia'].transform(
    lambda x: x.expanding().max()
)

# Crear tabla de stats del año anterior para merge
stats_prev = df[['delito', 'codcom', 'año', 'semana_numero', 'promedio_hist_anual', 'std_hist_anual', 'max_hist_anual']].copy()
stats_prev['año'] = stats_prev['año'] + 1
stats_prev = stats_prev.rename(columns={
    'promedio_hist_anual': 'promedio_hist_anual_prev',
    'std_hist_anual': 'std_hist_anual_prev',
    'max_hist_anual': 'max_hist_anual_prev'
})

# Merge y reemplazar
df = df.merge(stats_prev, on=['delito', 'codcom', 'año', 'semana_numero'], how='left')
df['promedio_hist_anual'] = df['promedio_hist_anual_prev'].fillna(df['promedio_hist_anual'])
df['std_hist_anual'] = df['std_hist_anual_prev'].fillna(df['std_hist_anual'])
df['max_hist_anual'] = df['max_hist_anual_prev'].fillna(df['max_hist_anual'])
df.drop(columns=['promedio_hist_anual_prev', 'std_hist_anual_prev', 'max_hist_anual_prev'], inplace=True)

# =========================================
# 9. VARIACIONES Y MÉTRICAS AVANZADAS
# =========================================
df['var_pct_vs_semana_anterior'] = (df['delta'] / df['casos_semana_anterior'].replace(0, np.nan)) * 100
df['z_score'] = (df['frecuencia'] - df['promedio_hist']) / df['std_hist'].replace(0, np.nan)
df['z_score_vs_año_anterior'] = (df['frecuencia'] - df['promedio_hist_anual']) / df['std_hist_anual'].replace(0, np.nan)
df['conclusion_z'] = pd.cut(df['z_score'].fillna(0), bins=[-np.inf, -2, 2, np.inf], labels=['Bajo', 'Normal', 'Alto'])
df['tendencia_corto_plazo'] = np.where(df['delta'] > 0, 'Alza', np.where(df['delta'] < 0, 'Baja', 'Estable'))

# Racha de semanas consecutivas en alza
df['racha'] = df.groupby(['delito', 'codcom'])['delta'].transform(
    lambda x: (x > 0).astype(int).groupby((x <= 0).cumsum()).cumsum()
)

# ID de la semana con máximo histórico
df['id_semana_max_hist'] = df.groupby(['delito', 'codcom'])['id_semana'].transform(
    lambda x: x.iloc[df.loc[x.index, 'frecuencia'].idxmax() - x.index[0]] if len(x) > 0 else np.nan
)

# Alertas
df['alerta_aumento_critico'] = (df['z_score'] > 2) & (df['var_pct_vs_semana_anterior'] > 30)
df['alerta_vs_año_anterior'] = (df['z_score_vs_año_anterior'] > 2) & (df['frecuencia'] > df['max_hist_anual'])

# Casos misma semana año anterior
df_prev_casos = df[['delito', 'codcom', 'año', 'semana_numero', 'frecuencia']].copy()
df_prev_casos['año'] = df_prev_casos['año'] + 1
df_prev_casos = df_prev_casos.rename(columns={'frecuencia': 'casos_misma_semana_año_anterior'})
df = df.merge(df_prev_casos, on=['delito', 'codcom', 'año', 'semana_numero'], how='left')

# =========================================
# 10. MERGE DATOS EXTERNOS
# =========================================
localiza = pd.read_excel(r"D:\GitHub\LOCALIZA_DB\Localiza Chile (1).xlsx")
localiza2 = localiza[['Provincia', 'Comuna', 'Región', 'Codcom', 'Codreg']].drop_duplicates()
df2 = df.merge(localiza2, left_on="codcom", right_on="Codcom")

df2 = df2.sort_values(['Codreg', 'delito', 'Codcom', 'id_semana'])
df2['ranking_comunal_regional'] = df2.groupby(['Codreg', 'delito', 'id_semana'])['frecuencia'].rank(method='dense', ascending=False)
df2['ranking_comunal_regional_semana_anterior'] = df2.groupby(['Codreg', 'delito', 'Codcom'])['ranking_comunal_regional'].shift(1)

clasePoblacion = pd.read_excel(r"C:\Users\limc_\Downloads\Factores Población.xlsx", sheet_name="Clase Población")
factor = pd.read_excel(r"C:\Users\limc_\Downloads\Factores Población.xlsx", sheet_name="Factores")

clasePoblacion2 = clasePoblacion[['Codcom', 'Población', 'Clase Población']]
clasePoblacion2.columns = ['Codcom', 'poblacion_clase', 'clase_poblacion']
factor2 = factor[['Codcom', 'Año', 'Población', 'Factor Población']]
factor2.columns = ['Codcom', 'año', 'poblacion', 'facor_poblacion']

df3 = df2.merge(clasePoblacion2).merge(factor2)
del df3["Codcom"]

print(f"Filas procesadas: {len(df3):,}")
print(f"Comunas únicas: {df3['codcom'].nunique()}")

In [ ]:
# =====================================================
# 11. CÁLCULOS PARA TARJETAS (Cards)
# =====================================================

# --- A. Proyecciones y Tasas ---
df3['semana_numero_safe'] = df3['semana_numero'].replace(0, 1)
df3['proyeccion_anual'] = (df3['acumulado_anual'] / df3['semana_numero_safe']) * 52
df3['tasa_semanal'] = (df3['frecuencia'] / df3['poblacion']) * 100000
df3['tasa_proyectada_anual'] = (df3['proyeccion_anual'] / df3['poblacion']) * 100000

# --- B. Agregaciones Nacionales ---
grp_nac = df3.groupby(['delito', 'id_semana'])
df3['tasa_proyectada_nacional'] = grp_nac['proyeccion_anual'].transform('sum') / grp_nac['poblacion'].transform('sum') * 100000
df3['tasa_semanal_nacional'] = grp_nac['frecuencia'].transform('sum') / grp_nac['poblacion'].transform('sum') * 100000

# --- C. Agregaciones Regionales ---
grp_reg = df3.groupby(['Codreg', 'delito', 'id_semana'])
df3['tasa_proyectada_regional'] = grp_reg['proyeccion_anual'].transform('sum') / grp_reg['poblacion'].transform('sum') * 100000
df3['tasa_semanal_regional'] = grp_reg['frecuencia'].transform('sum') / grp_reg['poblacion'].transform('sum') * 100000
df3['casos_semana_regional'] = grp_reg['frecuencia'].transform('sum')
df3['aporte_pct_region'] = (df3['frecuencia'] / df3['casos_semana_regional'].replace(0, np.nan)) * 100

# --- D. Rankings ---
df3['ranking_regional_proy_anual'] = df3.groupby(['Codreg', 'delito', 'id_semana'])['proyeccion_anual'].rank(method='dense', ascending=False)
df3['ranking_nacional_semanal'] = df3.groupby(['delito', 'id_semana'])['frecuencia'].rank(method='dense', ascending=False)
df3['ranking_nacional_proy_anual'] = df3.groupby(['delito', 'id_semana'])['proyeccion_anual'].rank(method='dense', ascending=False)

grp_cluster = df3.groupby(['clase_poblacion', 'delito', 'id_semana'])
df3['ranking_cluster_semanal'] = grp_cluster['frecuencia'].rank(method='dense', ascending=False)
df3['ranking_cluster_proy_anual'] = grp_cluster['proyeccion_anual'].rank(method='dense', ascending=False)

# Shifts de Rankings
df3 = df3.sort_values(['Codreg', 'delito', 'codcom', 'id_semana'])
g_temp = df3.groupby(['delito', 'codcom'])
df3['ranking_regional_proy_anual_anterior'] = g_temp['ranking_regional_proy_anual'].shift(1)
df3['ranking_nacional_semanal_anterior'] = g_temp['ranking_nacional_semanal'].shift(1)
df3['ranking_nacional_proy_anual_anterior'] = g_temp['ranking_nacional_proy_anual'].shift(1)
df3['ranking_cluster_semanal_anterior'] = g_temp['ranking_cluster_semanal'].shift(1)

# --- E. Stats Adicionales ---
df3['proyeccion_mes_actual'] = df3['media_movil_4s'] * 4.33
df3['promedio_diario_semanal'] = df3['frecuencia'] / 7
df3['promedio_diario_historico'] = df3['promedio_hist'] / 7

total_semanal_comuna = df3.groupby(['codcom', 'id_semana'])['frecuencia'].transform('sum')
df3['share_delito_semanal'] = (df3['frecuencia'] / total_semanal_comuna.replace(0, np.nan)) * 100

df3.drop(columns=['semana_numero_safe'], inplace=True, errors='ignore')

print("DataFrame Final Listo. Columnas:")
print(df3.columns.tolist())
print(f"\nTotal columnas: {len(df3.columns)}")

In [ ]:
# =========================================
# 12. VALIDACIÓN DETALLADA (SIMULACIÓN DASHBOARD)
# =========================================
# Filtramos Santiago, última semana disponible, delito Total
santiago = df3[(df3['codcom'] == 13101) & (df3['delito'] == 'Total')].sort_values('id_semana')
ultima = santiago.iloc[-1]

print(f"\n=== REPORTE DE VALIDACIÓN: {ultima['Comuna']} - {ultima['semana_detalle']} ===\n")

# TARJETA 1: Desempeño Semanal (Actual vs Anterior)
print(f"[T1] Semanal (Sem vs Sem Ant): {ultima['frecuencia']:.0f} vs {ultima['casos_semana_anterior']:.0f} | Var: {ultima['delta']:.0f}")

# TARJETA 2: Interanual (Actual vs Misma Sem Año Ant)
print(f"[T2] Interanual (Sem vs Sem Año Ant): {ultima['frecuencia']:.0f} vs {ultima['casos_misma_semana_año_anterior']:.0f}")

# TARJETA 3: Acumulado YTD
print(f"[T3] Acumulado YTD: {ultima['acumulado_anual']:.0f} vs {ultima['acumulado_anual_anterior']:.0f}")

# TARJETA 4: Proyección Anual
# Nota: Para validados T4 'Real Año Anterior' necesitaríamos sumar todo el año anterior. Simulamos valor aproximado.
print(f"[T4] Proyección Anual: {ultima['proyeccion_anual']:.0f} (estimado cierre)")

# TARJETA 5: Desviación Histórica
print(f"[T5] Histórico: Actual {ultima['frecuencia']:.0f} vs Promedio {ultima['promedio_hist']:.1f} | Z-Score: {ultima['z_score']:.2f}")

# TARJETA 6: Proyección Mensual
print(f"[T6] Proyección Mensual: {ultima['proyeccion_mes_actual']:.1f}")

# TARJETA 7: Media Móvil 4 Semanas
print(f"[T7] Media Móvil 4s: {ultima['media_movil_4s']:.1f}")

# TARJETA 8: Récords
print(f"[T8] Récords: Prom Diario {ultima['promedio_diario_semanal']:.1f} | Max Hist Semanal {ultima['max_hist']:.0f}")

# TARJETAS 9-12: Tasas x100k
print(f"\n--- Tasas x100k ---")
print(f"[T9] Tasa Proyectada Comunal: {ultima['tasa_proyectada_anual']:.1f}")
print(f"[T10] Tasa Proyectada Regional: {ultima['tasa_proyectada_regional']:.1f}")
print(f"[T11] Tasa Proyectada Nacional: {ultima['tasa_proyectada_nacional']:.1f}")
print(f"[T12] Tasa Semanal Comunal: {ultima['tasa_semanal']:.1f}")

# TARJETAS 13-18: Rankings
print(f"\n--- Rankings (Posición #1 = Peor/Más Delitos) ---")
print(f"[T13] Rank Regional Semanal: #{ultima['ranking_comunal_regional']:.0f} (Ant: #{ultima['ranking_comunal_regional_semana_anterior']:.0f})")
print(f"[T14] Rank Regional Anual: #{ultima['ranking_regional_proy_anual']:.0f} (Ant: #{ultima['ranking_regional_proy_anual_anterior']:.0f})")
print(f"[T15] Rank Nacional Semanal: #{ultima['ranking_nacional_semanal']:.0f} (Ant: #{ultima['ranking_nacional_semanal_anterior']:.0f})")
print(f"[T16] Rank Nacional Anual: #{ultima['ranking_nacional_proy_anual']:.0f} (Ant: #{ultima['ranking_nacional_proy_anual_anterior']:.0f})")
print(f"[T17] Rank Cluster Semanal: #{ultima['ranking_cluster_semanal']:.0f} (Ant: #{ultima['ranking_cluster_semanal_anterior']:.0f})")

# Validación de Lógica de Cambio de Ranking
diff = ultima['ranking_nacional_semanal_anterior'] - ultima['ranking_nacional_semanal']
# Si ranking anterior (e.g. 5) - actual (e.g. 2) = 3 -> Bajó de # -> Empeoró
# Si ranking anterior (e.g. 2) - actual (e.g. 5) = -3 -> Subió de # -> Mejoró
concl = 'MEJORA (Subió de puesto #)' if diff < 0 else 'EMPEORA (Bajó de puesto # acercándose al 1)' if diff > 0 else 'IGUAL'
print(f"Logic Check: Ranking Nacional cambió de {ultima['ranking_nacional_semanal_anterior']:.0f} a {ultima['ranking_nacional_semanal']:.0f} -> Diff: {diff} -> {concl}")

In [ ]:
# =========================================
# GUARDADO GLOBAL (data3.json.gz)
# =========================================
df3.to_json(r'D:\GitHub\STOP_WEB3\web_js\data\data3.json.gz', orient='records', compression='gzip', indent=None)
print("Archivo data3.json.gz guardado exitosamente.")

In [ ]:
# =========================================
# GUARDADO POR COMUNA (data/stop/{codcom})
# =========================================
import os
output_dir = r'D:\GitHub\STOP_WEB3\web_js\data\stop'
os.makedirs(output_dir, exist_ok=True)

for i in df3["codcom"].unique():
    aux = df3[df3["codcom"] == i]
    aux.to_json(fr'{output_dir}/{i}', orient='records', compression='gzip', indent=None)

print(f"Guardados {df3['codcom'].nunique()} archivos por comuna.")